# SAP NetWeaver RFC SDK Protocol (NWRFC)

The NetWeaver RFC SDK protocol is transported over SAP NI using the same RFC service ports as classic RFC/CPIC traffic. NWRFC frames can be identified by the `06 cb 02 00` magic bytes at the beginning of the NI payload, while classic SAPRFC frames use `06 03 02 00`.

The NWRFC payloads handled by pysap are TLV-oriented structures. Each TLV entry uses a two-byte big-endian tag, a two-byte big-endian length and a value. Most values are UTF-16LE strings; the password value is a binary `ab_scramble` field, and some function-call payloads contain ASCII XML fragments.

First we need to perform some setup to import the NWRFC helpers and small utilities used by the examples:

In [1]:
import struct
from pprint import pprint

from pysap.SAPNI import SAPNI
from pysap.SAPNWRFC import (
    NWRFC_MAGIC,
    SAPRFC_MAGIC,
    NWRFC_TAGS,
    NWRFC_USERNAME_TAGS,
    NWRFC_SID_RE,
    parse_tlv,
    decode_string,
    decode_value,
    extract_rfc_params,
    extract_xml_data,
)
from pysap.utils.crypto.rfc import ab_scramble, ab_descramble


def build_tlv(tag, value):
    return struct.pack(">HH", tag, len(value)) + value


def utf16(value):
    return value.encode("utf-16-le")


def describe_tlv_stream(data):
    rows = []
    for tag, raw in parse_tlv(data):
        rows.append({
            "tag": "0x%04x" % tag,
            "name": NWRFC_TAGS.get(tag, "unknown"),
            "length": len(raw),
            "value": "<password bytes>" if tag == 0x0117 else decode_value(raw),
        })
    return rows

## NWRFC frame inside SAP NI

An SAP NI record carries the NWRFC frame as its payload. The first four bytes of that payload distinguish NWRFC from classic SAPRFC/CPIC.

In [2]:
print("NWRFC magic:", NWRFC_MAGIC.hex(" "))
print("SAPRFC magic:", SAPRFC_MAGIC.hex(" "))

sample_frame = NWRFC_MAGIC + b"\x00\x00" + build_tlv(0x0114, utf16("100"))
ni_record = SAPNI() / sample_frame

print("NI payload length:", len(sample_frame))
print("Raw NI record:", bytes(ni_record).hex(" "))

NWRFC magic: 06 cb 02 00
SAPRFC magic: 06 03 02 00
NI payload length: 16
Raw NI record: 00 00 00 10 06 cb 02 00 00 00 01 14 00 06 31 00 30 00 30 00


## Connection and logon TLVs

Connection setup and logon frames include metadata such as the RFC destination, client IP address, hostname, program name, SAP client, language and username. Several username tags have been observed; pysap keeps them in priority order in `NWRFC_USERNAME_TAGS`.

In [3]:
logon_tlvs = b"".join([
    build_tlv(0x0006, utf16("BACKEND")),
    build_tlv(0x0007, utf16("10.0.0.10")),
    build_tlv(0x0008, utf16("sapnw752_NPL_00")),
    build_tlv(0x0100, utf16("pysap-demo")),
    build_tlv(0x0114, utf16("100")),
    build_tlv(0x0152, utf16("E")),
    build_tlv(0x0111, utf16("DEVELOPER")),
])

pprint(describe_tlv_stream(logon_tlvs))
print("Username tags in priority order:", ["0x%04x" % tag for tag in NWRFC_USERNAME_TAGS])

hostname = decode_string(dict(parse_tlv(logon_tlvs))[0x0008])
sid = NWRFC_SID_RE.search(hostname).group(1)
print("Extracted SID:", sid)

[{'length': 14, 'name': 'dest', 'tag': '0x0006', 'value': 'BACKEND'},
 {'length': 18, 'name': 'ip', 'tag': '0x0007', 'value': '10.0.0.10'},
 {'length': 30,
  'name': 'hostname',
  'tag': '0x0008',
  'value': 'sapnw752_NPL_00'},
 {'length': 20, 'name': 'program', 'tag': '0x0100', 'value': 'pysap-demo'},
 {'length': 6, 'name': 'client', 'tag': '0x0114', 'value': '100'},
 {'length': 2, 'name': 'language', 'tag': '0x0152', 'value': 'E'},
 {'length': 18, 'name': 'username', 'tag': '0x0111', 'value': 'DEVELOPER'}]
Username tags in priority order: ['0x0111', '0x0119', '0x0009']
Extracted SID: NPL


## Password TLV

The password field uses tag `0x0117`. Unlike most NWRFC TLVs, its value is binary: a four-byte little-endian seed followed by the `ab_scramble` result. For NWRFC captures, the scrambled password payload is UTF-16LE encoded.

In [4]:
password_value = ab_scramble("s3cr3t!", seed=0x12345678, encoding="utf-16-le")
password_tlv = build_tlv(0x0117, password_value)
parsed_password = dict(parse_tlv(password_tlv))[0x0117]

print("Password TLV length:", len(parsed_password))
print("Seed bytes:", parsed_password[:4].hex(" "))
print("Recovered password:", ab_descramble(parsed_password, encoding="utf-16-le"))

Password TLV length: 18
Seed bytes: 78 56 34 12
Recovered password: s3cr3t!


## RFC call parameter TLVs

Function call payloads can carry parameter names with tag `0x0201` and parameter values with tag `0x0203`. The helper `extract_rfc_params` scans for valid parameter names and decodes the immediately following value.

In [5]:
rfc_call = b"".join([
    b"\x00\x00" + build_tlv(0x0102, utf16("STFC_CONNECTION")),
    b"\x00\x00" + build_tlv(0x0201, utf16("REQUTEXT")),
    b"\x00\x00" + build_tlv(0x0203, utf16("Hello from pysap")),
])

pprint(describe_tlv_stream(rfc_call))
pprint(extract_rfc_params(rfc_call))

[{'length': 30,
  'name': 'function_module',
  'tag': '0x0102',
  'value': 'STFC_CONNECTION'},
 {'length': 16, 'name': 'param_name', 'tag': '0x0201', 'value': 'REQUTEXT'},
 {'length': 32,
  'name': 'param_value',
  'tag': '0x0203',
  'value': 'Hello from pysap'}]
{'REQUTEXT': 'Hello from pysap'}


## XML-serialized table and structure data

Some NWRFC function-call bodies serialize table and structure parameters as ASCII XML fragments inside the frame. The XML extraction helper returns scalar values as strings and table rows as lists.

In [6]:
xml_body = (
    b"\x06\xcb\x02\x00binary-prefix"
    b"<IV_GUID>abc123==</IV_GUID>"
    b"<IT_MODULE>"
    b"<item><FIELD>value1</FIELD></item>"
    b"<item><FIELD>value2</FIELD></item>"
    b"</IT_MODULE>"
)

pprint(extract_xml_data(xml_body))

{'IT_MODULE': [{'FIELD': 'value1'}, {'FIELD': 'value2'}], 'IV_GUID': 'abc123=='}


## Confirmed TLV tags

The parser recognizes the following tag names. Unknown tags may still appear in live captures, but pysap only assigns semantics to the confirmed tags below.

In [7]:
for tag, name in sorted(NWRFC_TAGS.items()):
    print("0x%04x  %s" % (tag, name))

0x0006  dest
0x0007  ip
0x0008  hostname
0x0009  username
0x0100  program
0x0102  function_module
0x0111  username
0x0114  client
0x0117  password
0x0119  username
0x0131  extended_passport
0x0152  language
0x0201  param_name
0x0203  param_value


## Extended Passport TLV

NWRFC uses confirmed tag `0x0131` for the same binary Extended Passport documented in the SAPEPP notebook. `parse_tlv` identifies the value and `decode_extended_passport` applies the shared packet model. This example covers the inner TLV only; WebSocket or other outer transports are outside the NWRFC helper's scope.

In [8]:
from pysap.SAPEPP import SAPEPP
from pysap.SAPNWRFC import decode_extended_passport

passport = SAPEPP(component=b"NWRFC", action=b"RFC_PING", client=b"100")
passport_tlv = build_tlv(0x0131, bytes(passport))
raw_passport = dict(parse_tlv(passport_tlv))[0x0131]
decoded_passport = decode_extended_passport(raw_passport)
assert bytes(decoded_passport) == bytes(passport)
print(NWRFC_TAGS[0x0131], decoded_passport.component.rstrip(b"\x00 "))

extended_passport b'NWRFC'
